In [8]:
import pandas as pd
import torch
import os
file_path = r'data\rawData'
txt_file_path = r'data\processedData'
new_path = r'd:\Desktop\PHD\reasearch\biyework\maml'
os.chdir(new_path)
# 检查当前路径是否切换成功
current_path = os.getcwd()
print(current_path)

d:\Desktop\PHD\reasearch\biyework\maml


In [9]:

# 读取CSV文件
csv_file_path = r'data\processedData\FLOOR3\all_data_new.csv'  # 替换为你的CSV文件路径
df = pd.read_csv(csv_file_path)
# 保存为PTH文件
pth_file_path = r"model\v1\input\FLOOR3_v4.pth"  
csv_file_path = r"model\v1\input\FLOOR3_v4.csv" #v4数据量史诗级攀升
pretrain_pth_path=r"model\v1\input\pretrain.pth"    # 只包含某一个SF的数据
finetune_pth_path=r"model\v1\input\finetune.pth"    # 包含pretrain里没有的sf的数据
test_pth_path=r"model\v1\input\test.pth"# 和finetune差不多

max_pretrain=500    # 预训练数据集每一个采样点最大样本数
max_finetune=100
max_test=100
pretrain_Sf=[7,9]
finetune_df_Sf=[8]
test_df_Sf=[8]




# todo: 可不可以预训练用的数据集和微调用的数据集是不同location_id的？如果这么做是否可以通过已知的location去预测未知location
# 读取 location_vector.csv 文件
location_vector_path = r'model\v1\output\location_vector_v3.csv'  # 替换为你的location_vector.csv文件路径，
location_df = pd.read_csv(location_vector_path)

# 创建 location_id 到 idx 的映射
location_id_to_idx = dict(zip(location_df['location_id'], location_df['idx']))
print(df['location_id'].unique())
print(location_id_to_idx)
# 将 location_id 转换为 idx，如果 location_id 不在映射中，则丢弃
df['location_id'] = df['location_id'].map(location_id_to_idx)
# 丢弃 location_id 为 NaN 的行
df = df.dropna(subset=['location_id'])
# 打印转换后的 location_id 列，检查是否有丢失的点
print("Mapped Location IDs:")
print(len(df['location_id'].unique()))
print(df['location_id'].unique())
# 检查并转换DataFrame中的数据类型
df = df.apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)
# 将sf全都除以12，将tp全都除以10
# todo：归一化是否要放到训练的时候做？归一化应该在某一个数据集（预训练、微调、测试）里做而不是整个数据集里做 以及SF tp是否要进行归一化 
df['sf'] = df['sf']
df['tp'] = df['tp']
df['rssi'] = df['rssi']
df['average_rssi'] = df['average_rssi'] 
# 将snr和rssi_variance归一化
df['snr'] = (df['snr'] - df['snr'].min()) / (df['snr'].max() - df['snr'].min())
# 确保数据集没有重叠
# 从原始数据中划分 pretrain 数据集
pretrain_df = df[df['sf'].isin(pretrain_Sf)]
pretrain_df = pretrain_df.groupby('location_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max_pretrain), random_state=42))
remaining_df = df.drop(pretrain_df.index)  # 从原始数据中移除 pretrain 数据

# 从剩余数据中划分 finetune 数据集
finetune_df = remaining_df[remaining_df['sf'].isin(finetune_df_Sf )]
finetune_df = finetune_df.groupby('location_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max_finetune),random_state=42))
remaining_df = remaining_df.drop(finetune_df.index)  # 从剩余数据中移除 finetune 数据

# 从剩余数据中划分 test 数据集
test_df = remaining_df[remaining_df['sf'].isin(test_df_Sf)]
test_df = test_df.groupby('location_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max_test),random_state=42))

# 检查是否有重叠
assert len(set(pretrain_df.index) & set(finetune_df.index)) == 0, "Pretrain and Finetune datasets overlap!"
assert len(set(pretrain_df.index) & set(test_df.index)) == 0, "Pretrain and Test datasets overlap!"
assert len(set(finetune_df.index) & set(test_df.index)) == 0, "Finetune and Test datasets overlap!"

# 保存数据集
torch.save({
    'rssi': torch.tensor(pretrain_df[['rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr"]].values, dtype=torch.float32),
    'sf': torch.tensor(pretrain_df["sf"].values, dtype=torch.float32),
    'tp': torch.tensor(pretrain_df['tp'].values, dtype=torch.float32),
    'snr': torch.tensor(pretrain_df[['sf', 'tp']].values, dtype=torch.float32),
    'label': torch.tensor(pretrain_df['location_id'].values, dtype=torch.int64)
}, pretrain_pth_path)

torch.save({
    'rssi': torch.tensor(finetune_df[['rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr"]].values, dtype=torch.float32),
    'sf': torch.tensor(finetune_df["sf"].values, dtype=torch.float32),
    'tp': torch.tensor(finetune_df['tp'].values, dtype=torch.float32),
    'snr': torch.tensor(finetune_df[['sf', 'tp']].values, dtype=torch.float32),
    'label': torch.tensor(finetune_df['location_id'].values, dtype=torch.int64)
}, finetune_pth_path)

torch.save({
    'rssi': torch.tensor(test_df[['rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr"]].values, dtype=torch.float32),
    'sf': torch.tensor(test_df["sf"].values, dtype=torch.float32),
    'tp': torch.tensor(test_df['tp'].values, dtype=torch.float32),
    'snr': torch.tensor(test_df[['sf', 'tp']].values, dtype=torch.float32),
    'label': torch.tensor(test_df['location_id'].values, dtype=torch.int64)
}, test_pth_path)
    

data_dict = {
    'rssi': torch.tensor(df[['rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr",'rssi','average_rssi','rssi_variance',"snr"]].values, dtype=torch.float32),
    'snr': torch.tensor(df[["sf","tp"]].values, dtype=torch.float32),
    'sf': torch.tensor(df['sf'].values, dtype=torch.float32),
    'tp': torch.tensor(df['tp'].values, dtype=torch.float32),
    'label':  torch.tensor(df['location_id'].values, dtype=torch.int64)
}

# todo 根据方差和平均值制造更多数据 得到一个rssi串（时序）来作为inputdim
# 确保数据长度是 batch_size * 2 * 16 的倍数
batch_size = 16  # 你可以根据需要调整 batch_size
total_length = len(df)
required_length = (total_length // (batch_size * 2 * 16)) * (batch_size * 2 * 16)

# 截断数据以匹配所需长度
data_dict['rssi'] = data_dict['rssi'][:required_length]
data_dict['snr'] = data_dict['snr'][:required_length]
data_dict['sf'] = data_dict['sf'][:required_length]
data_dict['tp'] = data_dict['tp'][:required_length]
data_dict['label'] = data_dict['label'][:required_length]
# pth的形式以字典的方式保存rssi,snr,sf,tp,location_id,label
torch.save(data_dict, pth_file_path)
# 保存同样的数据在csv文件中
df.to_csv(csv_file_path, index=False)
pretrain_df.to_csv(r"model\v1\input\pretrain.csv", index=False)
finetune_df.to_csv(r"model\v1\input\finetune.csv", index=False)
test_df.to_csv(r"model\v1\input\test.csv", index=False)

['1m' '302.0' '306-304' '308.0' '312' '322' '328' '330' '334' '336'
 '340.0' '344' '350' '354' '356.0' '360' '366.0' '370' 'point1' '308'
 '322.0' '340' '356' '302' '318' '348' '366' 'point2' 'point3' '310' '316'
 '320' '324' '326' '332' '338' '342' '346' '350.0' '352.0' '358' '362'
 '364' '368' '372']
{'point1': 0, '370': 1, '366': 2, '360': 3, '356': 4, '354': 5, '350': 6, '344': 7, '340': 8, '336': 9, '334': 10, '1m': 11, '330': 12, '328': 13, '322': 14, '312': 15, '308': 16, '306-304': 17, '302': 18}
Mapped Location IDs:
19
[11. 17. 15. 14. 13. 12. 10.  9.  7.  6.  5.  3.  1.  0. 16.  8.  4. 18.
  2.]


C:\Users\28052\AppData\Local\Temp\ipykernel_26608\1324733961.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pretrain_df = pretrain_df.groupby('location_id', group_keys=False).apply(lambda x: x.sample(n=min(len(x), max_pretrain), random_state=42))
C:\Users\28052\AppData\Local\Temp\ipykernel_26608\1324733961.py:56: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  finetune_df = finetune_df.groupby('location

In [10]:
# 验证一下这段代码
# 读取PTH文件
data_tensor = torch.load(pth_file_path, weights_only=True)
data_pretrain = torch.load(pretrain_pth_path, weights_only=True)
data_finetune = torch.load(finetune_pth_path, weights_only=True)
data_test = torch.load(test_pth_path, weights_only=True)
# 打印数据
# print("Data from PTH file:",pth_file_path)
# print(data_tensor['rssi'])
# print(data_tensor['rssi'].shape)
# print(data_tensor['snr'].shape)
# print(data_tensor['sf'].shape)
# print(data_tensor['tp'].shape)
# print(data_tensor['label'])

print("Data from PTH file:",pretrain_pth_path)
print(data_pretrain['rssi'].shape)
print(data_pretrain['sf'].unique()) 
print(data_pretrain['label'].unique())

print("Data from PTH file:",finetune_pth_path)
print(data_finetune['rssi'].shape)
print(data_finetune['sf'].unique())
print(data_finetune['label'].unique())

print("Data from PTH file:",test_pth_path)
print(data_test['rssi'].shape)
print(data_test['sf'].unique())
print(data_test['label'].unique())
# print(sorted(df['location_id'].unique()))
# print(df['sf'].unique())


Data from PTH file: model\v1\input\pretrain.pth
torch.Size([8704, 16])
tensor([7., 9.])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18])
Data from PTH file: model\v1\input\finetune.pth
torch.Size([1700, 16])
tensor([8.])
tensor([ 1,  2,  3,  4,  5,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
Data from PTH file: model\v1\input\test.pth
torch.Size([1700, 16])
tensor([8.])
tensor([ 1,  2,  3,  4,  5,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18])
